# DataBot — Agente RAG con Memoria Persistente (one tool calling)

### Requisitos previos antes de ejecutar este notebook
* Haber ejecutado el notebook que vectoriza el documento en embeddings
* Contar con credenciales de conexión a la tabla de embeddings y a la tabla que guarda las conversaciones para tener memoria.

---

Implementación de un chatbot con tres capacidades combinadas:

- **Tool Calling**: el LLM decide autónomamente si consultar la base de conocimiento
- **RAG (Retrieval-Augmented Generation)**: búsqueda semántica sobre documentos vectorizados en Supabase
- **Memoria persistente**: historial de conversación por sesión almacenado en PostgreSQL

## Flujo de la arquitectura

```mermaid
flowchart TD
    U([Usuario]) -->|"1 · Mensaje"| F["chat_con_agente()"]

    F -->|"2 · Carga historial de sesión"| PG[(PostgreSQL\nChat History)]
    PG -->|mensajes previos| F

    F -->|"3 · system_prompt + historial + mensaje"| LLM["GPT-4.1\n+ tools vinculadas"]

    LLM --> D{"¿Tool call?"}

    D -->|"No — saludo\nconversación general"| DIRECT["Respuesta directa"]

    D -->|"Sí — pregunta\nsobre DATAPATH"| TOOL["buscar_informacion()\nBase_de_conocimiento.py"]

    TOOL -->|"4a · query → embedding\ntext-embedding-ada-002"| OAIAPI["OpenAI\nEmbeddings API"]
    OAIAPI -->|"vector 1536-dim"| TOOL

    TOOL -->|"4b · SELECT * FROM tabla"| SB[("Supabase\nVectores")]
    SB -->|"docs + embeddings"| TOOL

    TOOL -->|"4c · Similitud coseno\ntop-5 docs relevantes"| LLM2["GPT-4.1\n2.ª llamada con contexto RAG"]

    LLM2 --> RESP2["Respuesta enriquecida\ncon información de DATAPATH"]

    DIRECT --> SAVE["Guardar turno en historial\n(PostgreSQL)"]
    RESP2 --> SAVE

    SAVE -->|"add_user_message\nadd_ai_message"| PG
    SAVE -->|"5 · Respuesta"| U
```

In [1]:
import os
import uuid
from urllib.parse import quote_plus

#from dotenv import load_dotenv, find_dotenv
#load_dotenv(find_dotenv())

#import sys
# Agregar el directorio raíz al path para importar tools
#sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_postgres import PostgresChatMessageHistory
import psycopg

In [ ]:
import sys

# Agregar la raíz del proyecto al path para importar "tools/" (2 niveles arriba del notebook)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Importar tools desde la carpeta "tools/"
from tools.Base_de_conocimiento import buscar_informacion

## Dependencias principales

| Módulo | Rol en el agente |
|--------|------------------|
| `init_chat_model` | Inicializa GPT-4.1 con interfaz unificada de LangChain |
| `HumanMessage / AIMessage / ToolMessage` | Tipado de mensajes en el ciclo de tool calling |
| `PostgresChatMessageHistory` | Persiste el historial de chat por `session_id` en PostgreSQL |
| `psycopg` | Driver de conexión síncrona a PostgreSQL |
| `buscar_informacion` | Tool RAG — convierte la consulta a embedding y recupera docs de Supabase |

## Conexión a PostgreSQL — Memoria del agente

El historial de conversación se persiste en PostgreSQL. Cada sesión se identifica con un **UUID único**, lo que permite reanudar conversaciones sin perder contexto e aislar múltiples usuarios en la misma tabla.

`quote_plus` codifica caracteres especiales en la contraseña antes de incluirla en la URL de conexión.

In [3]:
# ============================================
# Carga de variables de entorno para la conexión a PostgreSQL
# ============================================
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT", "5432")
DB_NAME     = os.getenv("DB_NAME", "postgres")

if not all([DB_USER, DB_PASSWORD, DB_HOST]):
    raise ValueError(
        "❌ Faltan variables de base de datos en .env\n"
        "Requeridas: DB_USER, DB_PASSWORD, DB_HOST\n"
        "Opcionales: DB_PORT (default: 5432), DB_NAME (default: postgres)"
    )

# quote_plus maneja caracteres especiales en la contraseña (@ # $ etc.)
DATABASE_URL = f"postgresql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"🔌 Conectando como: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

🔌 Conectando como: postgres.ebdkewopehhsvkhboofb@aws-1-us-east-1.pooler.supabase.com:5432/postgres


In [4]:
# ============================================
# CREAR TABLA DE HISTORIAL de texto en PostgreSQL
# (Se usará para almacenar el historial de chat entre el usuario y el bot)
# ============================================

# Nombre de la tabla en supabase para almacenar el historial textual de chat User-Bot
TBL_NAME_CHAT_USER_BOT = os.getenv("TBL_NAME_CHAT_USER_BOT")

def crear_tabla_historial(table_name: str = TBL_NAME_CHAT_USER_BOT):
    """Crea la tabla de historial en PostgreSQL si no existe."""
    try:
        sync_connection = psycopg.connect(DATABASE_URL)

        # Se crea tabla (si no existe) con la estructura necesaria 
        # para almacenar el historial de chat (creada por "PostgresChatMessageHistory")
        PostgresChatMessageHistory.create_tables(sync_connection, table_name)
        
        sync_connection.close()

        print(f"✅ Tabla '{table_name}' lista en PostgreSQL")
    except Exception as e:
        print(f"⚠️ Nota sobre tabla: {e}")

crear_tabla_historial()

✅ Tabla 'tbl_chat_history_text' lista en PostgreSQL


`PostgresChatMessageHistory.create_tables()` crea la tabla con el esquema esperado por LangChain si aún no existe — la operación es **idempotente**. Cada fila almacena: `session_id`, `type` (user / ai), `content` y timestamp.

La función `get_session_history(session_id)` actúa como **fábrica de historiales**: abre una conexión a PostgreSQL y devuelve el objeto `PostgresChatMessageHistory` con todos los mensajes previos de esa sesión, listos para ser inyectados al contexto del LLM.

In [ ]:
# ============================================
# FUNCIÓN QUE CARGA EL HISTÓRICO DE CONVERSACIÓN
# ============================================
def get_session_history(session_id: str, table_name: str = TBL_NAME_CHAT_USER_BOT) -> PostgresChatMessageHistory:
    sync_connection = psycopg.connect(DATABASE_URL)
    return PostgresChatMessageHistory(
        table_name,
        session_id,
        sync_connection=sync_connection
    )

## Herramientas del agente (Tool Calling)

La lista `tools` declara las capacidades externas que el LLM puede invocar. Al hacer `chat.bind_tools(tools)`, LangChain envía el **esquema JSON** de cada tool (nombre, descripción, parámetros) junto con cada request al modelo.

GPT-4.1 decide autónomamente si llamar a alguna tool o responder directamente, basándose en el `system_prompt` y el contexto del mensaje. En este agente la única tool registrada es `buscar_informacion`, que encapsula toda la lógica RAG definida en `tools/Base_de_conocimiento.py`.

In [6]:
# ============================================
# LISTA DE TOOLS DISPONIBLES
# ============================================
# Agregar aquí todas las tools que quieras usar
tools = [
    buscar_informacion, # <-- se deberá referenciar en el "system_prompt"
]

# En la tool "buscar_informacion" se realiza la búsqueda de información en la base de datos vectorial,
# la cual contiene información en embeddings creados en el notebook de vectorización de la base de conocimiento.

tools

[StructuredTool(name='buscar_informacion', description='Busca información sobre DATAPATH en la base de conocimientos.\nUsa esta herramienta cuando el usuario pregunte sobre:\n- Programas de DATAPATH\n- Cursos y contenidos\n- Docentes e instructores\n- Precios y modalidades\n- Cualquier información relacionada con DATAPATH\n\nArgs:\n    consulta: La pregunta o tema a buscar\n    TABLE_EMBEDDINGS: Nombre de la tabla de embeddings', args_schema=<class 'langchain_core.utils.pydantic.buscar_informacion'>, func=<function buscar_informacion at 0x71f9487319e0>)]

In [7]:
# ============================================
# CONFIGURACIÓN DEL MODELO CON TOOLS
# ============================================
chat = init_chat_model(
    "gpt-4.1", 
    temperature=0.4
)

chat_con_tools = chat.bind_tools(tools)
chat_con_tools

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10', 'langchain-openai': '1.3.2'}}, output_version=None, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x71f9471621d0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x71f947163190>, root_client=<openai.O

## System Prompt — Control de decisiones del agente

El `system_prompt` cumple dos roles críticos:

1. **Identidad y tono**: define al agente como *DataBot*, con instrucciones de responder en español de forma clara y amigable
2. **Routing de tools**: instruye explícitamente **cuándo usar** `buscar_informacion` y **cuándo no**, evitando consultas innecesarias a Supabase en turnos conversacionales (saludos, agradecimientos)

Este control condicional es esencial en agentes RAG: sin él, el modelo consultaría la base vectorial incluso para responder un simple *"hola"*, aumentando la latencia y el costo innecesariamente.

In [ ]:
# ============================================
# PROMPT DEL AGENTE en donde de le indica cuando usar las tools y cuando no
# ============================================
system_prompt = """Eres DataBot, un asistente de IA de DATAPATH.

Tu objetivo es ayudar a los usuarios respondiendo sus preguntas.

INSTRUCCIONES:
- Para preguntas sobre DATAPATH (programas, cursos, precios, docentes), USA la herramienta "buscar_informacion".
- Para saludos, agradecimientos o conversación general, responde directamente SIN usar herramientas.
- Recuerdas toda la conversación gracias a tu memoria persistente.
- Responde siempre en español de manera clara y amigable.

EJEMPLOS de cuándo NO usar herramientas (no se hacen preguntas sobre DATAPATH):
- "Hola" → Responde con un saludo
- "Gracias" → Responde amablemente
- "¿Cómo estás?" → Responde conversacionalmente

EJEMPLOS de cuándo SÍ usar la herramienta "buscar_informacion":
- "¿Qué cursos tienen?" → Usa la Tool "buscar_informacion"
- "¿Cuánto cuesta el programa de IA?" → Usa la Tool "buscar_informacion"
- "¿Quiénes son los docentes?" → Usa la Tool "buscar_informacion" """

## Lógica principal — `chat_con_agente()`

La función implementa el ciclo completo de un turno de conversación en **dos posibles caminos**:

### Camino 1 — Respuesta directa (sin tool)
```
mensaje → LLM (1.ª llamada) → respuesta → guardar en PostgreSQL
```

### Camino 2 — RAG con tool calling (pregunta sobre DATAPATH)
```
mensaje → LLM (1.ª llamada, emite tool_call)
        → buscar_informacion()
            → embed(query) con text-embedding-ada-002
            → SELECT * FROM Supabase
            → similitud coseno → top-5 docs
        → LLM (2.ª llamada con contexto RAG)
        → respuesta enriquecida → guardar en PostgreSQL
```

**Pasos internos:**
1. Carga el historial previo de PostgreSQL para el `session_id`
2. Construye el array de mensajes: `[system] + [historial] + [mensaje actual]`
3. Primera llamada al LLM — responde directamente o emite `tool_calls`
4. Si hay `tool_calls`: ejecuta cada tool, adjunta el `ToolMessage` con el resultado y hace una segunda llamada al LLM para generar la respuesta final con contexto RAG
5. Persiste el nuevo turno completo (`add_user_message` + `add_ai_message`) en PostgreSQL

In [ ]:
# ============================================
# 7. FUNCIÓN DE CHAT CON AGENTE + TOOLS
# ============================================
def chat_con_agente(mensaje_usuario: str, session_id: str) -> str:
    """
    Ejecuta el agente con tools y memoria.
    El agente decide si usar herramientas o responder directamente.
    """
    # ***********************************************************
    # Obtener historial
    # con esto el agente puede recordar la conversaciones previas con un mismo usuario
    history = get_session_history(session_id)
    mensajes_previos = history.messages
    
    # Construir mensajes para el modelo
    messages = [{"role": "system", "content": system_prompt}]
    
    # Agregar historial
    for msg in mensajes_previos:
        if isinstance(msg, HumanMessage):
            messages.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            messages.append({"role": "assistant", "content": msg.content})
    
    # Agregar mensaje actual
    messages.append({"role": "user", "content": mensaje_usuario})
    
    # ***********************************************************
    # Invocar modelo con tools
    # con esto, el agente decide si usar herramientas o responder directamente
    response = chat_con_tools.invoke(messages)
    
    # Procesar tool calls si existen
    if response.tool_calls:
        # Ejecutar cada tool
        tool_results = []
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            # Buscar y ejecutar la tool
            for t in tools:
                if t.name == tool_name:
                    result = t.invoke(tool_args)
                    tool_results.append({
                        "tool_call_id": tool_call["id"],
                        "result": result
                    })
                    break
        
        # Agregar respuesta del modelo con tool calls y resultados
        messages.append(response)
        for tr in tool_results:
            messages.append(ToolMessage(
                content=tr["result"],
                tool_call_id=tr["tool_call_id"]
            ))
        
        # Segunda llamada para obtener respuesta final
        final_response = chat_con_tools.invoke(messages)
        respuesta_final = final_response.content
    else:
        # Sin tool calls, respuesta directa
        respuesta_final = response.content
    
    # Guardar en historial
    history.add_user_message(mensaje_usuario)
    history.add_ai_message(respuesta_final)
    
    return respuesta_final

## Loop de conversación interactivo — `main()`

Implementa la interfaz de línea de comandos del agente:

- **Nueva sesión**: genera un UUID con `uuid.uuid4()` — cada conversación queda aislada en PostgreSQL
- **Reanudar sesión**: el usuario pega un UUID previo para retomar el contexto exacto donde lo dejó

El UUID es la **clave de partición** del historial — compartirlo permite continuar la misma conversación desde cualquier proceso o máquina que tenga acceso a la base de datos.

In [10]:
# ============================================
# 8. LOOP DE CONVERSACIÓN
# ============================================
def main():
    print("=" * 60)
    print("🤖 DataBot - Agente con TOOLS + MEMORIA PERSISTENTE")
    print("=" * 60)
    print("🔧 Tools disponibles:")
    for t in tools:
        print(f"   - {t.name}")
    print("💾 Historial: PostgreSQL")
    
    # Menú de sesión
    print("\nOpciones de sesión:")
    print("  1. Nueva conversación")
    print("  2. Continuar sesión existente (pegar UUID)")
    
    opcion = input("\nElige (1/2): ").strip()
    
    if opcion == "2":
        session_id = input("Pega el UUID de la sesión: ").strip()
        try:
            uuid.UUID(session_id)
        except ValueError:
            print("⚠️ UUID inválido. Creando nueva sesión...")
            session_id = str(uuid.uuid4())
    else:
        session_id = str(uuid.uuid4())
    
    print(f"\n📝 Session ID: {session_id}")
    print("   (Guarda este ID para continuar después)")
    print("✅ El agente DECIDE cuándo buscar en la base de conocimiento")
    print("Escribe 'salir' para volver al menú.\n")
    
    print("*" * 60)
    print("💬 Comienza a chatear con DataBot:")
    while True:
        usuario = input("Tú: ").strip()
        print(f"💬 Usuario: {usuario}")
        
        if usuario.lower() in ['salir', 'exit', 'quit']:
            print(f"\n💾 Tu sesión está guardada.")
            print(f"   UUID: {session_id}")
            print("👋 ¡Hasta luego!")
            break
        
        if not usuario:
            continue
        
        try:
            respuesta = chat_con_agente(usuario, session_id)
            print(f"🤖 DataBot: {respuesta}\n")
        except Exception as e:
            print(f"\n❌ Error: {e}\n")

In [12]:
# ============================================
# MAIN
# ============================================
main()

🤖 DataBot - Agente con TOOLS + MEMORIA PERSISTENTE
🔧 Tools disponibles:
   - buscar_informacion
💾 Historial: PostgreSQL

Opciones de sesión:
  1. Nueva conversación
  2. Continuar sesión existente (pegar UUID)

📝 Session ID: b73c002a-87ae-4405-a4cb-64e8257e9fd7
   (Guarda este ID para continuar después)
✅ El agente DECIDE cuándo buscar en la base de conocimiento
Escribe 'salir' para volver al menú.

************************************************************
💬 Comienza a chatear con DataBot:
💬 Usuario: quién eres?
🤖 DataBot: ¡Hola! Soy DataBot, el asistente de inteligencia artificial de DATAPATH. Estoy aquí para ayudarte a resolver tus dudas, brindarte información y acompañarte en lo que necesites relacionado con DATAPATH o simplemente para conversar. ¿En qué puedo ayudarte hoy?

💬 Usuario: de qué trata el curso?
   🔍 Buscando: 'De qué trata el curso'
🤖 DataBot: El curso de DATAPATH es un bootcamp intensivo enfocado en AI Engineering para desarrolladores. Aprenderás desde los fundamen

## Resumen de componentes

| Componente | Tecnología | Función |
|------------|-----------|---------|
| LLM | GPT-4.1 (OpenAI) | Razonamiento, decisión de tools y generación de respuestas |
| Embeddings | text-embedding-ada-002 | Convierte la query del usuario a vector para búsqueda semántica |
| Base vectorial | Supabase (pgvector) | Almacena los documentos de DATAPATH como embeddings |
| Memoria | PostgreSQL (`langchain-postgres`) | Persiste el historial de chat por sesión UUID |
| Tool | `buscar_informacion` | Orquesta el flujo RAG: embed → retrieve → rank por similitud coseno |
| Similitud | Coseno (numpy) | Mide la relevancia semántica entre la query y cada documento |